# Create the LCIs file for use with *premise*

This notebook imports the individual LCIs for lithium projects as provided in Schenker & Pfister (2025) and export them into a unique Excel file that can be used with premise. This is done for the baseline LCIs (average technology parameters) and the optimized LCIs (optimistic technology parameters)

Some modifications are done on the lithium LCIs:

- Update some metadata of activities like "flow" to "reference product", "amount" to "production amount", add "database", etc.
- Some process names are used across projects (e.g., df_DLE_evaporation_ponds) - problematic to have everything in the same database - create unique activity names by adding the project name
- Change activity names to ecoinvent convention
- Change electricity from high voltage to medium voltage
- Generate unique code with wurst
- Location of foreground dataset is the name of the project (e.g., Chaerhan) - change to ISO location
- Reference product is missing in activities and exchanges
- Link water flows to the new biosphere database with regionalized water flows
- Add "input" field for foreground processes
- Inputs of "heat production, natural gas, at industrial furnace >100kW" and "machine operation, diesel, >= 74.57 kW, high load factor" have same location as project due to regionalization - change location to RoW

In [1]:
import bw2data as bd
import bw2io as bi
import wurst
import pandas as pd
from pathlib import Path
import shutil
import copy
from utils import relink_to_regionalized_water

16:08:44+0100 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


c:\Users\istrateir\AppData\Local\miniconda3\envs\plca\Lib\site-packages\bw2calc\__init__.py:57: UserWarning: No fast sparse solver found
  warnings.warn("No fast sparse solver found")


In [2]:
BW_PROJECT = 'plca_lithium'
bd.projects.set_current(BW_PROJECT)
print(bd.databases)

LITHIUM_DB = "lithium_brine_projects"
ECOINVENT_DB = "ecoinvent-3.10.1-cutoff"
BIOSPHERE_DB = "ecoinvent-3.10.1-biosphere"
WATER_DB = "biosphere-3.10.1-water regionalized"

technosphere = lambda x: x["type"] == "technosphere"
biosphere = lambda x: x["type"] == "biosphere"
production = lambda x: x["type"] == "production"
economic_flows = lambda x: x["type"] in ["technosphere", "production"]

Databases dictionary with 24 objects, including:
	biosphere-3.10-water regionalized
	biosphere-3.10.1-water regionalized
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff
	ecoinvent-3.10.1-biosphere
	ecoinvent-3.10.1-cutoff
	ecoinvent-3.11-biosphere
	ecoinvent-3.11-cutoff
	ei_cutoff_3.10_remind_SSP2-NDC_2025_STEPS 2025-11-04
	ei_cutoff_3.10_remind_SSP2-NDC_2035_STEPS 2025-11-04
Use `list(this object)` to get the complete list.


## Import ecoinvent database and lithium LCIs

In [3]:
# Import ecoinvent database with wurst (i.e., list of dictionary, each dict being a dataset)
try:
  len(ei_db)
except NameError:
  ei_db = wurst.extract_brightway2_databases(ECOINVENT_DB)

# Import biosphere database
bio_db = [ds for ds in bd.Database(BIOSPHERE_DB)]

Getting activity data


100%|██████████| 23523/23523 [00:00<00:00, 295354.60it/s]


Adding exchange data to activities


100%|██████████| 743409/743409 [00:23<00:00, 32123.90it/s]


Filling out exchange data


100%|██████████| 23523/23523 [00:01<00:00, 14380.05it/s]


In [4]:
# import lithium LCIs
def import_lci_dataset(LCI_PATH, ei_name, bio_name):
    """
    return dictionary containing the LCI: key: name of the project \ value
    """
    lci_files = [file for file in LCI_PATH.glob("*.xlsx")] + [file for file in LCI_PATH.glob("*.xls")]
    lci_dict = {}
    for file in lci_files:
        lci = bi.ExcelImporter(file)
        lci.apply_strategies(verbose=False)
        lci.match_database(ei_name, fields=('name', 'reference product', 'unit', 'location'))
        lci.match_database(bio_name, fields=('name', 'unit', 'categories'))
        lci_dict.update({lci.db_name: lci.data})
    return lci_dict

LCI_PATH_BASELINE = Path("../inventories/baseline")
lci_baseline_dict = import_lci_dataset(LCI_PATH_BASELINE, ECOINVENT_DB, BIOSPHERE_DB)

LCI_PATH_OPTIMIZED = Path("../inventories/optimized")
lci_optimized_dict = import_lci_dataset(LCI_PATH_OPTIMIZED, ECOINVENT_DB, BIOSPHERE_DB)

Extracted 1 worksheets in 0.04 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.67 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.05 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.04 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.03 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.02 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.02 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Extracted 1 worksheets in 0.04 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


## Formating LCI datasets metadata

In [5]:
# Import lithium project information
lithium_projects = pd.read_excel(Path("../scenario_data/lithium_projects_list.xlsx"))
project_locations = dict(zip(lithium_projects["Project name"], lithium_projects["Location"]))
print(len(project_locations))
print(project_locations)

24
{'Salar de Cauchari-Olaroz': 'AR', 'Chaerhan': 'CN-NWG', 'Salar de Centenario': 'AR', 'East Taijinar': 'CN-NWG', 'Sal de los Angeles': 'AR', 'Salar del Hombre Muerto North': 'AR', 'Kachi': 'AR', 'Lakkor Tso': 'CN-NWG', 'Maricunga': 'CL', 'Salar de Pastos Grandes': 'AR', 'Pozuelos': 'AR', 'Qinghai Yiliping': 'CN-NWG', 'Sal de Vida': 'AR', 'Salar de Arizaro': 'AR', 'Salar de Atacama': 'CL', 'Salar de Olaroz': 'AR', 'Salar de Tolillar': 'AR', 'Fenix': 'AR', 'Salar del Rincon': 'AR', 'Silver Peak': 'US', 'Tres Quebradas': 'AR', 'Salar de Uyuni': 'BO', 'Upper Rhine Graben': 'DE', 'Zhabuye': 'CN-NWG'}


In [6]:
def format_lci_ds(lcis_dict_raw):
    # First drop projects that are not within the scenarios
    lcis_dict = {}
    for project in lcis_dict_raw:
        if project in project_locations.keys():
            lcis_dict[project] = lcis_dict_raw[project]

    # Format the remaining projects
    for project in lcis_dict:
        # Create unique names for datasets associated with the project name
        unique_names = {ds["name"]: ds["name"] + "_" + project for ds in lcis_dict[project]}

        # Some changes in datasets metadata
        for ds in lcis_dict[project]:
            if "flow" in ds:
                ds['reference product'] = ds.pop('flow')
            if "amount" in ds:
                ds['production amount'] = ds.pop('amount')
            ds["database"] = LITHIUM_DB

            # Change to unique activity name
            if ds["name"] in unique_names.keys():
                ds.update({"name": unique_names[ds["name"]]})

            # Generate unique code
            ds.update({"code": wurst.filesystem.get_uuid()})

            # Change location of activity
            if ds["location"] in project_locations.keys():
                ds.update({"location": project_locations[ds["location"]]})

        # Make some changes in the exchange flows:
        for ds in lcis_dict[project]:
            for exc in filter(economic_flows, ds["exchanges"]):

                # Add product and input to production exchanges:
                # "product" is used instead of "reference product" for wurst
                if exc in filter(production, ds["exchanges"]):
                    exc.update({
                        "product": ds["reference product"],
                        "input": (ds["database"], ds["code"])})

                # Change exchange name to unique name
                if exc["name"] in unique_names.keys():
                    exc.update({"name": unique_names[exc["name"]]})
                    
                if exc["location"] in project_locations.keys():
                    exc.update({"location": project_locations[exc["location"]]})

                # Change from high to medium voltage
                if "electricity, high voltage" in exc["name"]:
                    exc.update({"name": "market for electricity, medium voltage"})
                    exc["product"] = "electricity, medium voltage"
    return lcis_dict

lci_baseline_dict = format_lci_ds(lci_baseline_dict)
lci_optimized_dict = format_lci_ds(lci_optimized_dict)

**Link to background processes and biosphere database**

In [7]:
def find_no_ecoinvent_lcis(lcis_dict):
    # Find if there is any background dataset that is not available in ecoinvent
    list_of_foreground_ds = [ds["name"] for project in lcis_dict for ds in lcis_dict[project]]
    list_of_ei_ds = [(ds["name"], ds["location"]) for ds in bd.Database(ECOINVENT_DB)]

    no_ecoinvent_lcis = []
    for project in lcis_dict:
        for ds in lcis_dict[project]:
            for exc in filter(technosphere, ds["exchanges"]):
                # Do this only for background datasets; i.e., datasets that are not in the project associated datasets
                if exc["name"] not in list_of_foreground_ds:
                    exc_ds = [ds for ds in list_of_ei_ds if ds[0] == exc["name"] and ds[1] == exc["location"]]
                    if len(exc_ds) == 0:
                        no_ecoinvent_lcis.append((exc["name"], exc["location"]))

    no_ecoinvent_lcis = list(set(no_ecoinvent_lcis))
    return no_ecoinvent_lcis

no_ecoinvent_lcis_baseline = find_no_ecoinvent_lcis(lci_baseline_dict)
no_ecoinvent_lcis_optimized = find_no_ecoinvent_lcis(lci_optimized_dict)

In [8]:
no_ecoinvent_lcis_baseline

[('heat production, natural gas, at industrial furnace >100kW', 'CN-NWG'),
 ('market for wastewater, average', 'CN-NWG'),
 ('heat production, natural gas, at industrial furnace >100kW', 'AR'),
 ('market for wastewater, average', 'AR'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'CN-NWG'),
 ('heat production, natural gas, at industrial furnace >100kW', 'BO'),
 ('heat production, natural gas, at industrial furnace >100kW', 'CL'),
 ('heat production, natural gas, at industrial furnace >100kW', 'US'),
 ('heat production, natural gas, at industrial furnace >100kW', 'DE'),
 ('market for soda ash, light', 'GLO'),
 ('market for wastewater, average', 'CL'),
 ('market for wastewater, average', 'BO'),
 ('market for wastewater, average', 'US'),
 ('market for wastewater, average', 'DE'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'AR'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'CL'),
 ('machine operation, diesel, >= 74.57 kW, high load fact

In [9]:
no_ecoinvent_lcis_optimized

[('heat production, natural gas, at industrial furnace >100kW', 'CN-NWG'),
 ('market for wastewater, average', 'CN-NWG'),
 ('heat production, natural gas, at industrial furnace >100kW', 'AR'),
 ('market for wastewater, average', 'AR'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'CN-NWG'),
 ('heat production, natural gas, at industrial furnace >100kW', 'BO'),
 ('heat production, natural gas, at industrial furnace >100kW', 'CL'),
 ('heat production, natural gas, at industrial furnace >100kW', 'US'),
 ('heat production, natural gas, at industrial furnace >100kW', 'DE'),
 ('market for soda ash, light', 'GLO'),
 ('market for wastewater, average', 'CL'),
 ('market for wastewater, average', 'BO'),
 ('market for wastewater, average', 'US'),
 ('market for wastewater, average', 'DE'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'AR'),
 ('machine operation, diesel, >= 74.57 kW, high load factor', 'CL'),
 ('machine operation, diesel, >= 74.57 kW, high load fact

In [10]:
def update_background_inputs(lcis_dict):
    # Flatten all foreground datasets once
    foreground_lookup = {d["name"]: d for project in lcis_dict for d in lcis_dict[project]}

    # Build a fast lookup for the background database once
    background_lookup = {(ds["name"], ds["location"], ds["unit"]): ds for ds in bd.Database(ECOINVENT_DB)}

    for project in lcis_dict:
        
        for ds in lcis_dict[project]:
            # Add product and input to technosphere exchanges:
            for exc in filter(technosphere, ds["exchanges"]):

                # Find datasets for foreground exchanges
                if exc["name"] in foreground_lookup:
                    try:
                        exc_ds = [foreground_lookup[exc["name"]]]
                    except KeyError:
                        exc_ds = []
                else:
                    # apply location/name adjustments first
                    special_location_map = {
                        "heat production, natural gas, at industrial furnace >100kW": "RoW",
                        "market for wastewater, average": "RoW",
                        "market for soda ash, light": "RoW",
                    }
                    if exc["name"] in special_location_map:
                        exc.update({"location": special_location_map[exc["name"]]})

                    if exc["name"] == "machine operation, diesel, >= 74.57 kW, high load factor":
                        exc.update({"location": "GLO"})

                    if exc["name"] == "deep well drilling, for deep geothermal power reg":
                        exc.update({"name": "deep well drilling, for deep geothermal power"})
                        if exc["location"] == "US-WECC":
                            exc.update({"location": "US-HICC"})

                    # Change sodium hydroxide generic market to market for sodium hydroxide
                    if exc["name"] == "sodium hydroxide to generic market for neutralising agent":
                        exc.update({
                            "name": "market for sodium hydroxide, without water, in 50% solution state",
                            "product": "sodium hydroxide, without water, in 50% solution state",
                            "location": "RoW"})

                    exc_ds = [background_lookup.get((exc["name"], exc["location"], exc["unit"]))]
                    exc_ds = [ds for ds in exc_ds if ds]  # drop None

                if len(exc_ds) > 1:
                    raise ValueError("More than one dataset for exchange", (exc["name"], exc["location"])) 
                            
                if len(exc_ds) == 0:
                    raise ValueError("LCI dataset not found for", (exc["name"], exc["location"]))
                        
                # "product" is used instead of "reference product" for wurst
                exc.update({
                    "product": exc_ds[0]["reference product"],
                    "input": (exc_ds[0]["database"], exc_ds[0]["code"])})
    return lcis_dict

lci_baseline_dict = update_background_inputs(lci_baseline_dict)
lci_optimized_dict = update_background_inputs(lci_optimized_dict)

In [11]:
def find_no_biosphere_flows(lci_dicts):
    # Find biosphere flows that are not in the biosphere database
    list_of_bio_ds = [(ds["name"], ds["categories"]) for ds in bd.Database(BIOSPHERE_DB)]
    no_biosphere_flows = []
    for project in lci_dicts:
        for ds in lci_dicts[project]:
            for exc in filter(biosphere, ds["exchanges"]):
                if (exc["name"], exc["categories"])  not in list_of_bio_ds:
                    no_biosphere_flows.append(exc["name"])
    return no_biosphere_flows

no_biosphere_flow_baseline = find_no_biosphere_flows(lci_baseline_dict)
no_biosphere_flow_optimized = find_no_biosphere_flows(lci_optimized_dict)

In [12]:
print(list(set(no_biosphere_flow_baseline)))
print(list(set(no_biosphere_flow_optimized)))

['Sodium']
['Sodium']


In [13]:
def update_biosphere_flows(lcis_dict):
    # Update "Sodium" to "Sodium I"
    sodium_i_ds = [ds for ds in bd.Database(BIOSPHERE_DB) if ds["name"]=="Sodium I" and ds["categories"]==("water",)][0]

    for project in lcis_dict:
        for ds in lcis_dict[project]:
            for exc in filter(biosphere, ds["exchanges"]):
                if exc["name"] == "Sodium":
                    exc["name"] = "Sodium I"
                    exc["input"] = sodium_i_ds.key
    return lcis_dict

lci_baseline_dict = update_biosphere_flows(lci_baseline_dict)
lci_optimized_dict = update_biosphere_flows(lci_optimized_dict)

### Link water flows to regionalized biosphere

In [14]:
water_flows_regionalized = [act for act in bd.Database(WATER_DB)]

for project in lci_baseline_dict:
    for ds in lci_baseline_dict[project]:
        relink_to_regionalized_water(water_flows_regionalized, ds, location=project)

for project in lci_optimized_dict:
    for ds in lci_optimized_dict[project]:
        relink_to_regionalized_water(water_flows_regionalized, ds, location=project)

### Export LCIs

In [15]:
lithium_dbs_all = [LITHIUM_DB, LITHIUM_DB + "-optimized"]
counter=0
for sc_db in [lci_baseline_dict, lci_optimized_dict]:
    lci_all = []
    for project in sc_db:
        lci_all.extend(sc_db[project])
    if lithium_dbs_all[counter] in bd.databases:
        del bd.databases[lithium_dbs_all[counter]]
    wurst.write_brightway2_database(lci_all, lithium_dbs_all[counter])
    counter+=1

Graph statistics for `lithium_brine_projects` importer:
432 graph nodes:
	process: 432
2235 graph edges:
	technosphere: 1426
	production: 432
	biosphere: 377
2235 edges to the following databases:
	lithium_brine_projects: 1100
	ecoinvent-3.10.1-cutoff: 758
	ecoinvent-3.10.1-biosphere: 296
	biosphere-3.10.1-water regionalized: 81
0 unique unlinked edges (0 total):


16:13:09+0100 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 432/432 [00:00<00:00, 3063.05it/s]


16:13:26+0100 [info     ] Vacuuming database            
Created database: lithium_brine_projects
Graph statistics for `lithium_brine_projects-optimized` importer:
432 graph nodes:
	process: 432
2235 graph edges:
	technosphere: 1426
	production: 432
	biosphere: 377
2235 edges to the following databases:
	lithium_brine_projects-optimized: 1100
	ecoinvent-3.10.1-cutoff: 758
	ecoinvent-3.10.1-biosphere: 296
	biosphere-3.10.1-water regionalized: 81
0 unique unlinked edges (0 total):


16:14:58+0100 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 432/432 [00:00<00:00, 1686.98it/s]


16:15:10+0100 [info     ] Vacuuming database            
Created database: lithium_brine_projects-optimized


In [16]:
# Export inventories to Excel file
for db in lithium_dbs_all:
    export_path = bi.export.excel.write_lci_excel(db)
    filepath = Path("../inventories")
    shutil.copy(export_path, filepath)